# Mobile Game LTV Forecasting

This notebook builds a Lifetime Value (LTV) prediction model for mobile game users.

## Objectives
1. Explore and understand user behavior patterns
2. Build predictive models for LTV at different time horizons
3. Identify key drivers of user value
4. Provide actionable insights for user acquisition and monetization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

## 1. Data Loading and Exploration

In [ ]:
# Load the data
df = pd.read_csv('../data/mobile_game_users.csv', parse_dates=['install_date'])
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Data types and info
df.info()

In [ ]:
# Summary statistics
df.describe()

## 2. Key Metrics Overview

In [ ]:
# Key business metrics
print("="*50)
print("KEY BUSINESS METRICS")
print("="*50)
print(f"\nTotal Users: {len(df):,}")
print(f"Payer Conversion Rate: {df['is_payer'].mean()*100:.2f}%")
print(f"\nRevenue Metrics:")
print(f"  Total Revenue: ${df['total_revenue'].sum():,.2f}")
print(f"  ARPU (Avg Revenue Per User): ${df['total_revenue'].mean():.2f}")
print(f"  ARPPU (Avg Revenue Per Paying User): ${df[df['is_payer']==1]['total_revenue'].mean():.2f}")
print(f"\nRetention Rates:")
print(f"  Day 1: {df['d1_retention'].mean()*100:.1f}%")
print(f"  Day 7: {df['d7_retention'].mean()*100:.1f}%")
print(f"  Day 30: {df['d30_retention'].mean()*100:.1f}%")
print(f"\nEngagement Metrics:")
print(f"  Avg Sessions: {df['total_sessions'].mean():.1f}")
print(f"  Avg Session Duration: {df['avg_session_duration_min'].mean():.1f} min")
print(f"  Avg Playtime: {df['total_playtime_hours'].mean():.1f} hours")

## 3. Revenue Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Revenue distribution (all users)
axes[0].hist(df['total_revenue'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Total Revenue ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Revenue Distribution (All Users)')
axes[0].axvline(df['total_revenue'].mean(), color='red', linestyle='--', label=f'Mean: ${df["total_revenue"].mean():.2f}')
axes[0].legend()

# Revenue distribution (payers only)
payers = df[df['is_payer'] == 1]
axes[1].hist(payers['total_revenue'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Total Revenue ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Revenue Distribution (Payers Only)')
axes[1].axvline(payers['total_revenue'].mean(), color='red', linestyle='--', label=f'Mean: ${payers["total_revenue"].mean():.2f}')
axes[1].legend()

# Revenue by percentile (whale analysis)
payer_revenue_sorted = payers['total_revenue'].sort_values(ascending=False).reset_index(drop=True)
cumulative_pct = payer_revenue_sorted.cumsum() / payer_revenue_sorted.sum() * 100
axes[2].plot(range(1, len(cumulative_pct)+1), cumulative_pct)
axes[2].axhline(80, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Top N Payers')
axes[2].set_ylabel('Cumulative Revenue %')
axes[2].set_title('Whale Concentration (Pareto Analysis)')

plt.tight_layout()
plt.show()

## 4. Segmentation Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by Country
country_stats = df.groupby('country').agg({
    'total_revenue': 'mean',
    'is_payer': 'mean',
    'user_id': 'count'
}).rename(columns={'user_id': 'users', 'is_payer': 'payer_rate'})
country_stats = country_stats.sort_values('total_revenue', ascending=True)

axes[0, 0].barh(country_stats.index, country_stats['total_revenue'], color='steelblue')
axes[0, 0].set_xlabel('Avg Revenue ($)')
axes[0, 0].set_title('Average Revenue by Country')

# Revenue by Platform
platform_stats = df.groupby('platform')['total_revenue'].agg(['mean', 'sum', 'count'])
axes[0, 1].bar(platform_stats.index, platform_stats['mean'], color=['#4CAF50', '#2196F3'])
axes[0, 1].set_ylabel('Avg Revenue ($)')
axes[0, 1].set_title('Average Revenue by Platform')
for i, (idx, row) in enumerate(platform_stats.iterrows()):
    axes[0, 1].text(i, row['mean'] + 0.1, f'n={int(row["count"]):,}', ha='center')

# Revenue by Acquisition Source
source_stats = df.groupby('acquisition_source')['total_revenue'].mean().sort_values(ascending=True)
axes[1, 0].barh(source_stats.index, source_stats.values, color='coral')
axes[1, 0].set_xlabel('Avg Revenue ($)')
axes[1, 0].set_title('Average Revenue by Acquisition Source')

# Payer Rate by Acquisition Source
payer_rate = df.groupby('acquisition_source')['is_payer'].mean().sort_values(ascending=True) * 100
axes[1, 1].barh(payer_rate.index, payer_rate.values, color='mediumpurple')
axes[1, 1].set_xlabel('Payer Rate (%)')
axes[1, 1].set_title('Payer Conversion by Acquisition Source')

plt.tight_layout()
plt.show()

## 5. Engagement vs Monetization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sessions vs Revenue
session_bins = pd.cut(df['total_sessions'], bins=[0, 5, 10, 25, 50, 100, 500], labels=['1-5', '6-10', '11-25', '26-50', '51-100', '100+'])
session_revenue = df.groupby(session_bins)['total_revenue'].mean()
axes[0].bar(session_revenue.index.astype(str), session_revenue.values, color='teal')
axes[0].set_xlabel('Total Sessions')
axes[0].set_ylabel('Avg Revenue ($)')
axes[0].set_title('Revenue by Session Count')
axes[0].tick_params(axis='x', rotation=45)

# Retention impact on LTV
retention_ltv = pd.DataFrame({
    'Day 1 Retained': df.groupby('d1_retention')['ltv_day365'].mean(),
    'Day 7 Retained': df.groupby('d7_retention')['ltv_day365'].mean(),
    'Day 30 Retained': df.groupby('d30_retention')['ltv_day365'].mean()
}).T
retention_ltv.columns = ['Not Retained', 'Retained']
retention_ltv.plot(kind='bar', ax=axes[1], color=['#ff6b6b', '#51cf66'])
axes[1].set_ylabel('Avg LTV ($)')
axes[1].set_title('LTV by Retention Status')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Status')

# Tutorial completion impact
tutorial_impact = df.groupby('tutorial_completed').agg({
    'total_revenue': 'mean',
    'is_payer': 'mean'
})
x = np.arange(2)
width = 0.35
axes[2].bar(x - width/2, tutorial_impact['total_revenue'], width, label='Avg Revenue ($)', color='steelblue')
ax2 = axes[2].twinx()
ax2.bar(x + width/2, tutorial_impact['is_payer'] * 100, width, label='Payer Rate (%)', color='coral')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Not Completed', 'Completed'])
axes[2].set_ylabel('Avg Revenue ($)', color='steelblue')
ax2.set_ylabel('Payer Rate (%)', color='coral')
axes[2].set_title('Tutorial Completion Impact')

plt.tight_layout()
plt.show()

## 6. Feature Engineering for LTV Prediction

In [ ]:
# Create a copy for modeling
df_model = df.copy()

# Encode categorical variables
label_encoders = {}
categorical_cols = ['acquisition_source', 'country', 'platform', 'age_group']

for col in categorical_cols:
    le = LabelEncoder()
    df_model[f'{col}_encoded'] = le.fit_transform(df_model[col])
    label_encoders[col] = le

# Create engagement score
df_model['engagement_score'] = (
    df_model['d1_retention'] * 0.1 +
    df_model['d7_retention'] * 0.3 +
    df_model['d30_retention'] * 0.6 +
    np.log1p(df_model['total_sessions']) * 0.1 +
    df_model['tutorial_completed'] * 0.2
)

# Sessions per day
df_model['sessions_per_day'] = df_model['total_sessions'] / np.maximum(df_model['days_since_install'], 1)

# Define features for modeling
feature_cols = [
    # Encoded categoricals
    'acquisition_source_encoded', 'country_encoded', 'platform_encoded', 'age_group_encoded',
    # Engagement
    'd1_retention', 'd7_retention', 'd30_retention',
    'total_sessions', 'avg_session_duration_min', 'total_playtime_hours',
    'levels_completed', 'tutorial_completed', 'ads_watched',
    'friends_invited', 'guild_joined',
    # Derived features
    'engagement_score', 'sessions_per_day'
]

print(f"Features for modeling: {len(feature_cols)}")
print(feature_cols)

## 7. LTV Prediction Model

In [ ]:
# Prepare features and target
X = df_model[feature_cols]
y = df_model['ltv_day365']  # Predict 365-day LTV

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = []

for name, model in models.items():
    # Train
    if 'Regression' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

    print(f"{name}:")
    print(f"  MAE: ${mae:.2f}")
    print(f"  RMSE: ${rmse:.2f}")
    print(f"  R2: {r2:.4f}")
    print()

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('LTV Prediction - Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

## 8. Model Predictions Analysis

In [ ]:
# Use the best model (Gradient Boosting) for final predictions
best_model = models['Gradient Boosting']
y_pred_final = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_final, alpha=0.5, s=10)
max_val = max(y_test.max(), y_pred_final.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
axes[0].set_xlabel('Actual LTV ($)')
axes[0].set_ylabel('Predicted LTV ($)')
axes[0].set_title('Actual vs Predicted LTV')
axes[0].legend()

# Residuals distribution
residuals = y_test - y_pred_final
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')

plt.tight_layout()
plt.show()

## 9. Business Insights and Recommendations

In [ ]:
# LTV by cohort analysis
df_model['predicted_ltv'] = best_model.predict(X)

# High value user identification
ltv_percentiles = df_model['predicted_ltv'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])
print("LTV Percentiles:")
for pct, val in ltv_percentiles.items():
    print(f"  {int(pct*100)}th percentile: ${val:.2f}")

# Segment users
def segment_user(ltv):
    if ltv == 0:
        return 'Non-Payer'
    elif ltv < ltv_percentiles[0.75]:
        return 'Low Value'
    elif ltv < ltv_percentiles[0.95]:
        return 'Medium Value'
    else:
        return 'High Value (Whale)'

df_model['user_segment'] = df_model['predicted_ltv'].apply(segment_user)

print("\nUser Segments:")
segment_summary = df_model.groupby('user_segment').agg({
    'user_id': 'count',
    'predicted_ltv': ['mean', 'sum']
}).round(2)
segment_summary.columns = ['Count', 'Avg LTV', 'Total LTV']
segment_summary['% of Users'] = (segment_summary['Count'] / len(df_model) * 100).round(2)
segment_summary['% of Revenue'] = (segment_summary['Total LTV'] / segment_summary['Total LTV'].sum() * 100).round(2)
print(segment_summary)

In [ ]:
# Actionable insights
print("="*60)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("="*60)

# Best acquisition sources
best_sources = df_model.groupby('acquisition_source')['predicted_ltv'].mean().sort_values(ascending=False)
print("\n1. ACQUISITION OPTIMIZATION")
print(f"   Best performing channels: {best_sources.index[0]} (${best_sources.iloc[0]:.2f} avg LTV)")
print(f"   Consider increasing spend on: {', '.join(best_sources.index[:3])}")

# Best countries
best_countries = df_model.groupby('country')['predicted_ltv'].mean().sort_values(ascending=False)
print("\n2. GEO TARGETING")
print(f"   Highest LTV countries: {', '.join(best_countries.index[:3])}")
print(f"   Top country avg LTV: ${best_countries.iloc[0]:.2f}")

# Platform insights
platform_ltv = df_model.groupby('platform')['predicted_ltv'].mean()
print("\n3. PLATFORM STRATEGY")
print(f"   iOS avg LTV: ${platform_ltv.get('iOS', 0):.2f}")
print(f"   Android avg LTV: ${platform_ltv.get('Android', 0):.2f}")

# Engagement impact
print("\n4. ENGAGEMENT PRIORITIES")
top_features = feature_importance.tail(5)['feature'].tolist()
print(f"   Focus on improving: {', '.join(top_features)}")

# Tutorial impact
tutorial_ltv = df_model.groupby('tutorial_completed')['predicted_ltv'].mean()
tutorial_lift = (tutorial_ltv[1] / tutorial_ltv[0] - 1) * 100 if tutorial_ltv[0] > 0 else 0
print(f"\n5. ONBOARDING")
print(f"   Tutorial completion increases LTV by {tutorial_lift:.1f}%")
print(f"   Prioritize tutorial optimization and completion rates")

## 10. Save Model and Predictions

In [ ]:
import joblib

# Save the best model
joblib.dump(best_model, '../data/ltv_model.pkl')
joblib.dump(scaler, '../data/scaler.pkl')
joblib.dump(label_encoders, '../data/label_encoders.pkl')

# Save predictions
df_model[['user_id', 'predicted_ltv', 'user_segment']].to_csv('../data/ltv_predictions.csv', index=False)

print("Model and predictions saved successfully!")
print("\nFiles saved:")
print("  - ../data/ltv_model.pkl")
print("  - ../data/scaler.pkl")
print("  - ../data/label_encoders.pkl")
print("  - ../data/ltv_predictions.csv")